# Globális lakásvásárlási döntés — bináris klasszifikáció

## Cél és kérdés

Ebben a notebookban azt vizsgáljuk, hogy a lakás, a környék és a vevő pénzügyi
adatai alapján előre jelezhető-e a `decision` változó értéke. A pozitív osztály
(`decision = 1`) azt jelenti, hogy a megfigyelés a pozitív döntési kategóriába
tartozik; a konkrét üzleti jelentést a Kaggle-adatleírás alapján kell kezelni,
mert az adat nem tartalmaz részletes döntési szótárt.

A cél nem egyetlen magas pontszám elérése, hanem egy szivárgásmentes,
érthető modell-összehasonlítás. A hamis pozitív és a hamis negatív hiba költsége
nincs számszerűsítve, ezért a rangsorolási képességet (`PR-AUC`) és a két osztály
kiegyensúlyozott felismerését (`balanced accuracy`) előre választjuk fő
mutatónak. A küszöböt később a tanító adatokból, üzleti költség hiányában a
balanced accuracy maximuma alapján választjuk ki.


In [ ]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    cross_validate,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
DATA_PATH = Path("global_house_purchase_dataset_clean.csv")
assert DATA_PATH.exists(), f"Nem található az adatfájl: {DATA_PATH.resolve()}"

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid", context="notebook")


## 1. Nyers adat, célváltozó és EDA

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")
ratio_raw = df["emi_to_income_ratio"].astype("string")
ratio_numeric = pd.to_numeric(ratio_raw, errors="coerce")
excel_date_mask = ratio_numeric.isna() & ratio_raw.str.match(r"^\d{1,2}\.", na=False)
ratio_numeric.loc[excel_date_mask] = (
    ratio_raw.loc[excel_date_mask].str.extract(r"^(\d{1,2})\.")[0].astype(float) / 100
)
if ratio_numeric.isna().any():
    bad_values = ratio_raw.loc[ratio_numeric.isna()].unique().tolist()
    raise ValueError(f"Nem értelmezhető emi_to_income_ratio értékek: {bad_values[:10]}")
df["emi_to_income_ratio"] = ratio_numeric

print(f"Méret: {df.shape[0]:,} sor × {df.shape[1]} oszlop")
display(df.head())
display(df.dtypes.to_frame("dtype"))


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()
target_counts = df["decision"].value_counts().sort_index()
target_rate = df["decision"].mean()

print("Hiányzó értékek:")
display(missing[missing > 0].to_frame("missing_count"))
print(f"Teljesen duplikált sorok: {duplicates:,}")
print(f"Pozitív osztály aránya: {target_rate:.1%}")
display(target_counts.rename(index={0: "decision=0", 1: "decision=1"}).to_frame("count"))
print("Célváltozó-ellenőrzés: néhány bemeneti változó erős szabályos kapcsolatot mutat.")
display(
    df.groupby(["legal_cases_on_property", "satisfaction_score"], observed=True)["decision"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "positive_rate"})
    .tail(10)
)


### EDA-megfigyelések és hipotézisek

1. A célváltozó aránya nem teljesen kiegyensúlyozott, ezért a nyers accuracy
   félrevezető lehet.
2. A `price`, `loan_amount`, `down_payment` és `customer_salary` együtt
   pénzügyi terheltséget jelezhetnek; ezért a hányadosok informatívabbak
   lehetnek, mint az egyes változók külön-külön.
3. A `satisfaction_score`, `neighbourhood_rating` és `connectivity_score`
   magasabb értékei összefügghetnek a pozitív döntéssel.
4. A kategóriák (`country`, `city`, `property_type`, `furnishing_status`)
   eltérő csoportkockázatokat rejthetnek, ezért one-hot kódolást használunk.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="decision", ax=axes[0], color="#3973ac")
axes[0].set_title("Célváltozó megoszlása")
axes[0].set_xlabel("decision")
axes[0].set_ylabel("Darabszám")

sns.boxplot(data=df, x="decision", y="price", showfliers=False, ax=axes[1], color="#66a61e")
axes[1].set_title("Ár döntési osztály szerint")
axes[1].set_ylabel("Ár")

sns.boxplot(data=df, x="decision", y="emi_to_income_ratio", showfliers=False, ax=axes[2], color="#e6ab02")
axes[2].set_title("EMI/jövedelem arány")
axes[2].set_ylabel("Arány")

plt.tight_layout()
plt.show()


In [ ]:
rate_by_country = (
    df.groupby("country", observed=True)["decision"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=rate_by_country.reset_index(),
    x="country",
    y="mean",
    color="#756bb1",
    ax=ax,
)
ax.axhline(target_rate, color="black", linestyle="--", label="teljes pozitív arány")
ax.set_title("Pozitív döntési arány országonként")
ax.set_ylabel("Pozitív arány")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=35)
ax.legend()
plt.tight_layout()
plt.show()
display(rate_by_country.rename(columns={"mean": "positive_rate"}))


In [ ]:
numeric_for_corr = df.select_dtypes(include=np.number).drop(columns=["property_id"])
corr = numeric_for_corr.corr(numeric_only=True)["decision"].drop("decision").sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=corr.values, y=corr.index, color="#d95f02", ax=ax)
ax.set_title("Numerikus változók korrelációja a céllal")
ax.set_xlabel("Pearson-korreláció")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 2. Tisztítás és a split

Hiányzó érték nincs, ezért imputálás várhatóan nem szükséges, de a pipeline-okban
szerepel, hogy a megoldás robusztus maradjon új adatok esetén. A CSV-ben az
`emi_to_income_ratio` 1760 értéke Excel-dátumként jelent meg (például
`03.jan`), holott a környezetében arányként szereplő `0.03` értéket jelenti.
Ezeket a nap-részt századdá alakítottuk, és az átalakítást dokumentáltuk.
Az
`property_id` csak azonosító, ezért elhagyjuk: az egyedi azonosító megtanulása
nem általánosítható. Az `emi_to_income_ratio` szöveges formátumát már a split
előtt numerikussá alakítottuk, ami nem adatból tanuló művelet.

A teszthalmazt most zároljuk. A későbbi modell- és feature-választás kizárólag
a tanító adaton történik. A tanító részen belül külön validációs halmazt
használunk a hangolás utáni, még nem látott ellenőrzéshez.


In [ ]:
target = "decision"
drop_columns = ["property_id", target]
X = df.drop(columns=drop_columns)
y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_dev, X_validation, y_dev, y_validation = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    stratify=y_train,
    random_state=RANDOM_STATE,
)

print("Tanító:", X_train.shape, "Teszt:", X_test.shape)
print("Fejlesztési:", X_dev.shape, "Hangolási ellenőrzés:", X_validation.shape)
print(f"Teszt pozitív arány: {y_test.mean():.1%}")


## 3. Előkészítés és alapmodellek

In [ ]:
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=np.number).columns.tolist()

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)
categorical_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features),
        ("categorical", categorical_preprocessor, categorical_features),
    ]
)

def make_model(estimator, **params):
    return Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )

models = {
    "dummy": make_model(DummyClassifier(strategy="prior", random_state=RANDOM_STATE)),
    "logistic_regression": make_model(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "decision_tree": make_model(
        DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )
    ),
    "random_forest": make_model(
        RandomForestClassifier(
            n_estimators=180,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
def cv_summary(model, X_data, y_data, cv_strategy=cv):
    scoring = {
        "pr_auc": "average_precision",
        "balanced_accuracy": "balanced_accuracy",
        "precision": "precision",
        "recall": "recall",
    }
    result = cross_validate(
        model,
        X_data,
        y_data,
        scoring=scoring,
        cv=cv_strategy,
        n_jobs=-1,
        return_train_score=False,
    )
    return {
        metric: (result[f"test_{metric}"].mean(), result[f"test_{metric}"].std())
        for metric in scoring
    }

model_rows = []
for name, model in models.items():
    summary = cv_summary(model, X_dev, y_dev)
    row = {"model": name}
    for metric, (mean, std) in summary.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    model_rows.append(row)

model_comparison = pd.DataFrame(model_rows).sort_values("pr_auc_mean", ascending=False)
display(model_comparison.round(3))


In [ ]:
best_baseline_name = model_comparison.iloc[0]["model"]
best_baseline = models[best_baseline_name]

oof_prob = cross_val_predict(
    best_baseline,
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

print(f"CV-ben legjobb alapmodell: {best_baseline_name}")
print(f"OOF PR-AUC: {average_precision_score(y_dev, oof_prob):.3f}")


## 4. Visszacsatolási kör: új, két változóból képzett jellemzők

Az EDA alapján azt a hipotézist teszteljük, hogy a puszta ár vagy hitelösszeg
helyett az arányok írják le jobban a pénzügyi terhelést. A feature engineering
nem használja a célváltozót, ezért minden foldban ugyanúgy reprodukálható.
Az összehasonlítást ugyanazokkal a foldokkal végezzük, így a páros különbség
értelmezhetőbb, mint két különböző felosztás eredménye.


In [ ]:
def add_engineered_features(data):
    result = data.copy()
    eps = 1e-9
    result["price_per_sqft"] = result["price"] / result["property_size_sqft"].clip(lower=eps)
    result["loan_to_price"] = result["loan_amount"] / result["price"].clip(lower=eps)
    result["down_payment_to_price"] = result["down_payment"] / result["price"].clip(lower=eps)
    result["expenses_to_salary"] = result["monthly_expenses"] / result["customer_salary"].clip(lower=eps)
    result["down_payment_share_of_finance"] = result["down_payment"] / (
        result["down_payment"] + result["loan_amount"]
    ).clip(lower=eps)
    result["ratings_mean"] = result[
        ["satisfaction_score", "neighbourhood_rating", "connectivity_score"]
    ].mean(axis=1)
    return result.replace([np.inf, -np.inf], np.nan)

X_dev_fe = add_engineered_features(X_dev)
X_validation_fe = add_engineered_features(X_validation)
X_train_fe = add_engineered_features(X_train)
X_test_fe = add_engineered_features(X_test)

numeric_features_fe = X_dev_fe.select_dtypes(include=np.number).columns.tolist()
categorical_features_fe = X_dev_fe.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
preprocessor_fe = ColumnTransformer(
    transformers=[
        ("numeric", numeric_preprocessor, numeric_features_fe),
        ("categorical", categorical_preprocessor, categorical_features_fe),
    ]
)

def make_fe_model(estimator):
    return Pipeline(steps=[("preprocess", preprocessor_fe), ("model", estimator)])

fe_models = {
    "logistic_regression_fe": make_fe_model(
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "random_forest_fe": make_fe_model(
        RandomForestClassifier(
            n_estimators=180,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ),
}

fe_rows = []
for name, model in fe_models.items():
    summary = cv_summary(model, X_dev_fe, y_dev)
    row = {"model": name}
    for metric, (mean, std) in summary.items():
        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std
    fe_rows.append(row)

fe_comparison = pd.DataFrame(fe_rows).sort_values("pr_auc_mean", ascending=False)
display(fe_comparison.round(3))


In [ ]:
base_oof = cross_val_predict(
    models["random_forest"],
    X_dev,
    y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]
fe_oof = cross_val_predict(
    fe_models["random_forest_fe"],
    X_dev_fe,
    y_dev,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

paired_difference = average_precision_score(y_dev, fe_oof) - average_precision_score(y_dev, base_oof)
print(f"Random Forest feature engineering páros OOF PR-AUC különbsége: {paired_difference:+.3f}")
print("A különbség akkor meggyőző, ha a foldonkénti eredmények szórásához képest is érdemi.")

fold_differences = []
for fold_number, (train_idx, valid_idx) in enumerate(cv.split(X_dev, y_dev), start=1):
    base_fold = models["random_forest"].fit(X_dev.iloc[train_idx], y_dev.iloc[train_idx])
    fe_fold = fe_models["random_forest_fe"].fit(X_dev_fe.iloc[train_idx], y_dev.iloc[train_idx])
    base_score = average_precision_score(
        y_dev.iloc[valid_idx],
        base_fold.predict_proba(X_dev.iloc[valid_idx])[:, 1],
    )
    fe_score = average_precision_score(
        y_dev.iloc[valid_idx],
        fe_fold.predict_proba(X_dev_fe.iloc[valid_idx])[:, 1],
    )
    fold_differences.append(
        {"fold": fold_number, "base_pr_auc": base_score, "fe_pr_auc": fe_score, "difference": fe_score - base_score}
    )
fold_difference_table = pd.DataFrame(fold_differences)
display(fold_difference_table.round(3))
print(
    "Páros különbség átlag ± szórás: "
    f"{fold_difference_table['difference'].mean():+.3f} ± "
    f"{fold_difference_table['difference'].std(ddof=1):.3f}"
)


## 5. Hangolás csak a fejlesztési adaton

In [ ]:
rf_search = RandomizedSearchCV(
    estimator=fe_models["random_forest_fe"],
    param_distributions={
        "model__n_estimators": [60, 90, 120],
        "model__max_depth": [None, 8, 12, 18],
        "model__min_samples_leaf": [2, 5, 10, 20],
        "model__max_features": ["sqrt", 0.5],
    },
    n_iter=3,
    scoring="average_precision",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
)
rf_search.fit(X_dev_fe, y_dev)
print("Legjobb paraméterek:")
print(rf_search.best_params_)
print(f"Fejlesztési CV PR-AUC: {rf_search.best_score_:.3f}")


In [ ]:
validation_models = {
    "baseline_random_forest": models["random_forest"],
    "tuned_random_forest_fe": rf_search.best_estimator_,
}
validation_rows = []
for name, model in validation_models.items():
    if name == "tuned_random_forest_fe":
        model.fit(X_dev_fe, y_dev)
        probabilities = model.predict_proba(X_validation_fe)[:, 1]
    else:
        model.fit(X_dev, y_dev)
        probabilities = model.predict_proba(X_validation)[:, 1]
    validation_rows.append(
        {
            "model": name,
            "validation_pr_auc": average_precision_score(y_validation, probabilities),
        }
    )

validation_comparison = pd.DataFrame(validation_rows).sort_values(
    "validation_pr_auc", ascending=False
)
display(validation_comparison.round(3))
selected_model_name = validation_comparison.iloc[0]["model"]
print(f"Előre nem látott validációs adaton kiválasztva: {selected_model_name}")


## 6. Küszöb kiválasztása a tanító OOF-jóslataiból

A PR-AUC küszöbfüggetlen mutató. A végső osztálycímkéhez viszont kell küszöb.
Üzleti költség hiányában nem a tesztadatból választunk, hanem a teljes tanító
halmazon készített out-of-fold jóslatok balanced accuracy maximumát keressük.


In [ ]:
final_model_template = rf_search.best_estimator_ if selected_model_name == "tuned_random_forest_fe" else models["random_forest"]
X_train_for_model = X_train_fe if selected_model_name == "tuned_random_forest_fe" else X_train
X_dev_for_model = X_dev_fe if selected_model_name == "tuned_random_forest_fe" else X_dev
X_test_for_model = X_test_fe if selected_model_name == "tuned_random_forest_fe" else X_test

train_oof_prob = cross_val_predict(
    final_model_template,
    X_train_for_model,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1,
)[:, 1]

thresholds = np.linspace(0.05, 0.95, 181)
threshold_rows = [
    {
        "threshold": threshold,
        "balanced_accuracy": balanced_accuracy_score(y_train, train_oof_prob >= threshold),
        "precision": precision_score(y_train, train_oof_prob >= threshold, zero_division=0),
        "recall": recall_score(y_train, train_oof_prob >= threshold, zero_division=0),
    }
    for threshold in thresholds
]
threshold_table = pd.DataFrame(threshold_rows)
chosen_threshold = float(
    threshold_table.loc[
        threshold_table["balanced_accuracy"].idxmax(), "threshold"
    ]
)
print(f"Rögzített küszöb a záró teszthez: {chosen_threshold:.3f}")
display(threshold_table.sort_values("balanced_accuracy", ascending=False).head())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_table["threshold"], threshold_table["balanced_accuracy"], label="balanced accuracy")
ax.plot(threshold_table["threshold"], threshold_table["precision"], label="precision")
ax.plot(threshold_table["threshold"], threshold_table["recall"], label="recall")
ax.axvline(chosen_threshold, color="black", linestyle="--", label="választott küszöb")
ax.set_title("Küszöbhatás tanító OOF-jóslatokon")
ax.set_xlabel("Küszöb")
ax.set_ylabel("Mutató")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Záró teszt — pontosan egyszer

In [ ]:
final_model = final_model_template.fit(X_train_for_model, y_train)
test_prob = final_model.predict_proba(X_test_for_model)[:, 1]
test_pred = (test_prob >= chosen_threshold).astype(int)

test_metrics = {
    "PR-AUC": average_precision_score(y_test, test_prob),
    "balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
}
top_fraction = 0.10
top_n = int(len(test_prob) * top_fraction)
top_risk_precision = y_test.iloc[np.argsort(-test_prob)[:top_n]].mean()
test_metrics["top_10_percent_precision"] = top_risk_precision
display(pd.Series(test_metrics, name="test_value").to_frame().round(3))

print("Konfúziós mátrix:")
display(
    pd.DataFrame(
        confusion_matrix(y_test, test_pred),
        index=["valós 0", "valós 1"],
        columns=["jósolt 0", "jósolt 1"],
    )
)
print(classification_report(y_test, test_pred, digits=3, zero_division=0))

precision_curve, recall_curve, curve_thresholds = precision_recall_curve(y_test, test_prob)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall_curve, precision_curve, label=f"modell PR-AUC = {test_metrics['PR-AUC']:.3f}")
ax.axhline(y_test.mean(), color="gray", linestyle="--", label="pozitív arány baseline")
ax.set_title("Precision–recall görbe a zárolt teszthalmazon")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_scores = []
positive_indices = np.flatnonzero(y_test.to_numpy() == 1)
negative_indices = np.flatnonzero(y_test.to_numpy() == 0)

for _ in range(300):
    sample_indices = rng.integers(0, len(y_test), len(y_test))
    sample_y = y_test.to_numpy()[sample_indices]
    if sample_y.min() == sample_y.max():
        continue
    bootstrap_scores.append(
        average_precision_score(sample_y, test_prob[sample_indices])
    )

ci_low, ci_high = np.percentile(bootstrap_scores, [2.5, 97.5])
print(
    f"Teszt PR-AUC bootstrap 95%-os konfidenciaintervalluma: "
    f"{test_metrics['PR-AUC']:.3f} [{ci_low:.3f}, {ci_high:.3f}]"
)


## 8. Korlátok

1. A `decision` változó jelentése és a kétféle hiba valós pénzügyi költsége
   nincs részletesen megadva, ezért a küszöb csak módszertani választás, nem
   végleges üzleti döntés.
2. A célváltozó több bemeneti változóval nagyon erős, szabályos kapcsolatot
   mutat (például a jogi esetek és az elégedettségi pontszám kombinációja).
   Ez lehet a szintetikus adat generálásának szabálya, de ha ilyen változók a
   döntés után keletkeztek, akkor éles környezetben cél-szivárgást jelentenének.
3. A véletlenszerű, rétegzett split feltételezi, hogy a jövőbeli adatok
   hasonló eloszlásúak; időbeli vagy országonkénti eltolódás esetén az eredmény
   optimista lehet.
4. A dataset egyetlen forrásból származik, ezért a jó tesztmutató nem bizonyítja
   a más országokra, időszakokra vagy más adatgyűjtési folyamatokra való
   általánosíthatóságot.

## Összegzés

A végső modellt nem a tesztpontszám alapján választottuk. A döntés a tanító
adatok CV-eredményeire és az elkülönített validációra épült, a teszthalmazt
pedig csak egyszer, a végső becsléshez használtuk fel. A notebook minden
adatból tanuló lépése pipeline-ban fut, így az imputálás, skálázás és one-hot
kódolás nem szivárog át a validációs vagy tesztadatból.
